In [0]:
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
import warnings; warnings.filterwarnings("ignore")

np.random.seed(42)

# Fresh pull (schema may have changed since last run)
pdf = spark.table("datathon.belmonte_hunters.alpha_dataset").toPandas()
pdf["datetime_utc"]   = pd.to_datetime(pdf["datetime_utc"])
pdf["datetime_local"] = pd.to_datetime(pdf["datetime_local"])
pdf["target_date"]    = pd.to_datetime(pdf["target_date"])

# Folds (same as before) — redefined here so this cell is self-contained
folds = [
    (pd.Timestamp("2025-04-01"), pd.Timestamp("2025-04-08"), pd.Timestamp("2025-05-27")),
    (pd.Timestamp("2025-05-20"), pd.Timestamp("2025-05-27"), pd.Timestamp("2025-07-15")),
    (pd.Timestamp("2025-07-08"), pd.Timestamp("2025-07-15"), pd.Timestamp("2025-09-02")),
    (pd.Timestamp("2025-08-26"), pd.Timestamp("2025-09-02"), pd.Timestamp("2025-10-21")),
    (pd.Timestamp("2025-10-14"), pd.Timestamp("2025-10-21"), pd.Timestamp("2025-12-03")),
]

# Diagnostics
print(f"rows={len(pdf):,}  communities={pdf['community_code'].nunique()}  "
      f"span {pdf['target_date'].min().date()} → {pdf['target_date'].max().date()}")
print("\nNaN rate (key cols):")
print(pdf[["alpha_lag_7d","n_clients_at_cutoff","D_mw","total_kw",
           "temperature_2m","is_holiday"]].isna().mean().round(3))
print("\nRows per community:")
print(pdf.groupby("community_code").size().sort_values())
print("\nNegative/zero total_kw:", int((pdf["total_kw"] <= 0).sum()))

rows=489,408  communities=18  span 2025-01-29 → 2025-11-30

NaN rate (key cols):
alpha_lag_7d           0.000
n_clients_at_cutoff    0.000
D_mw                   0.000
total_kw               0.000
temperature_2m         0.001
is_holiday             0.000
dtype: float64

Rows per community:
community_code
CE     5952
ML    17856
CN    25048
AR    29368
CL    29368
CB    29368
GA    29368
AS    29368
MD    29368
RI    29368
CM    29372
AN    29372
MC    29372
EX    29372
NC    29372
CT    29372
PV    29372
VC    29372
dtype: int64

Negative/zero total_kw: 32


In [0]:
# ============================================================
# CELL 1 — Minimal features
# ============================================================
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import mean_absolute_error
TAU = 2*np.pi

d = pdf.dropna(subset=["alpha_lag_7d"]).copy()
d["datetime_utc"]   = pd.to_datetime(d["datetime_utc"])
d["target_date"]    = pd.to_datetime(d["target_date"])

# --- Target ---
eps = 1.0
d["baseline_kw"] = (d["alpha_lag_7d"] * d["n_clients_at_cutoff"] * d["D_mw"]).clip(lower=eps)
d["y_log"]       = np.log(d["total_kw"].clip(lower=eps)) - np.log(d["baseline_kw"])
d["baseline_log"] = np.log(d["baseline_kw"])

# --- Robust lag ---
d["alpha_lag_median4"] = d[["alpha_lag_7d","alpha_lag_14d","alpha_lag_21d","alpha_lag_28d"]].median(axis=1)

# --- Time-of-week (672 bins, 15-min × week) ---
d["qhour_of_week"] = (d["dow"]*96 + d["mod"]).astype("int16")

# --- Year seasonality (1 harmonic) ---
d["sin_doy_1"] = np.sin(TAU*d["doy"]/365.25)
d["cos_doy_1"] = np.cos(TAU*d["doy"]/365.25)

# --- Weather (HDD/CDD only) ---
d["hdd18"] = (18.0 - d["temperature_2m"]).clip(lower=0)
d["cdd22"] = (d["temperature_2m"] - 22.0).clip(lower=0)

# --- Holiday typology ---
def hol_type(row):
    if row["is_holiday"] != 1: return "none"
    m, dd = row["target_date"].month, row["target_date"].day
    if (m==4 and 13<=dd<=21) or (m==3 and dd>=25): return "easter"
    if (m==5 and dd<=8) or (m==4 and dd>=28):      return "puente"
    if m==8 and 10<=dd<=20:                        return "assumption"
    if (m==12 and dd>=20) or (m==1 and dd<=7):     return "xmas_reyes"
    return "regular"
d["hol_type"] = d.apply(hol_type, axis=1)

# --- Holiday proximity + bridge + mismatch (all from one daily table) ---
hd = (d.groupby(["community_code","target_date"], as_index=False)["is_holiday"]
        .max().sort_values(["community_code","target_date"]))
hd["hol_7d_ago"] = hd.groupby("community_code")["is_holiday"].shift(7).fillna(0).astype(int)
hd["hol_prev"]   = hd.groupby("community_code")["is_holiday"].shift(1).fillna(0)
hd["hol_next"]   = hd.groupby("community_code")["is_holiday"].shift(-1).fillna(0)
hd["is_bridge"]  = ((hd["hol_next"]==1) | (hd["hol_prev"]==1)).astype(int)

def _prox(g):
    hol = g["is_holiday"].values.astype(bool); n = len(hol)
    since = np.full(n, 30); to = np.full(n, 30)
    last = -99
    for i in range(n):
        if hol[i]: last = i
        if last >= 0: since[i] = min(i - last, 30)
    nxt = 10**9
    for i in range(n-1, -1, -1):
        if hol[i]: nxt = i
        if nxt < 10**8: to[i] = min(nxt - i, 30)
    g = g.copy(); g["days_since_hol"] = since; g["days_to_hol"] = to
    return g
hd = hd.groupby("community_code", group_keys=False).apply(_prox)

d = d.merge(hd[["community_code","target_date","hol_7d_ago","is_bridge",
                "days_since_hol","days_to_hol"]],
            on=["community_code","target_date"], how="left")
d["hol_mismatch_7d"] = (d["is_holiday"] != d["hol_7d_ago"]).astype(int)

# --- Cast categoricals ---
FEATS = ["alpha_lag_median4","sn_alpha_last","sn_alpha_mean_7d","baseline_log",
         "qhour_of_week","sin_doy_1","cos_doy_1",
         "hol_type","hol_mismatch_7d","is_bridge","days_to_hol","days_since_hol",
         "hdd18","cdd22","community_code"]
CATS = ["qhour_of_week","hol_type","hol_mismatch_7d","is_bridge","community_code"]
for c in CATS: d[c] = d[c].astype("category")

print("shape:", d.shape, "features:", len(FEATS))

shape: (489304, 55) features: 15


In [0]:
# ============================================================
# CELL 2 — Walk-forward validation (single global model)
# ============================================================
PARAMS = dict(
    objective="regression_l1", metric="mae",
    learning_rate=0.03, num_leaves=127, min_data_in_leaf=300,
    feature_fraction=0.85, bagging_fraction=0.85, bagging_freq=5,
    lambda_l2=1.0, n_estimators=3000, verbose=-1,
)

oof = []
for train_end, val_start, val_end in folds:
    tr = d[d["target_date"] < train_end]
    va = d[(d["target_date"] >= val_start) & (d["target_date"] < val_end)]

    m = lgb.LGBMRegressor(**PARAMS)
    m.fit(tr[FEATS], tr["y_log"],
          sample_weight=tr["baseline_kw"].values,
          eval_set=[(va[FEATS], va["y_log"])],
          eval_sample_weight=[va["baseline_kw"].values],
          categorical_feature=CATS,
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])

    log_r = np.clip(m.predict(va[FEATS]), -0.5, 0.5)
    v = va[["datetime_utc","community_code","baseline_kw","total_kw"]].copy()
    v["pred_kw"] = (v["baseline_kw"] * np.exp(log_r)).clip(lower=0)

    a = v.groupby("datetime_utc").agg(a=("total_kw","sum"), p=("pred_kw","sum"),
                                      b=("baseline_kw","sum"))
    mb, mm = mean_absolute_error(a["a"],a["b"]), mean_absolute_error(a["a"],a["p"])
    print(f"{val_start.date()}→{val_end.date()}: base={mb:,.0f}  model={mm:,.0f}  Δ={(1-mm/mb)*100:.1f}%")
    oof.append(v)

oof = pd.concat(oof, ignore_index=True)
ao = oof.groupby("datetime_utc").agg(a=("total_kw","sum"), p=("pred_kw","sum"))
print(f"\nFINAL OOF MAE: {mean_absolute_error(ao['a'], ao['p']):,.0f}")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[127]	valid_0's l1: 0.123397
2025-04-08→2025-05-27: base=20,153  model=19,391  Δ=3.8%
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[6]	valid_0's l1: 0.0869477
2025-05-27→2025-07-15: base=14,333  model=13,289  Δ=7.3%
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 0.0936148
2025-07-15→2025-09-02: base=16,605  model=16,548  Δ=0.3%
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1]	valid_0's l1: 0.0866515
2025-09-02→2025-10-21: base=12,723  model=12,558  Δ=1.3%
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[24]	valid_0's l1: 0.0822656
2025-10-21→2025-12-03: base=15,745  model=13,319  Δ=15.4%

FINAL OOF MAE: 15,078
